In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import sklearn
import joblib
import fastapi

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("fastapi:", fastapi.__version__)

Python: 3.12.10
pandas: 3.1.0
numpy: 2.5.4
matplotlib: 3.11.2
seaborn: 0.13.2
scikit-learn: 1.9.1
joblib: 1.6.0
fastapi: 0.159.1


In [2]:
import pandas as pd

df = pd.read_csv('../datasets/raw/student-mat.csv', sep=';')

def risk_label(g3):
    if g3 < 10:
        return 'at_risk'
    elif g3 <= 12:
        return 'needs_review'
    else:
        return 'on_track'

df['risk_label'] = df['G3'].apply(risk_label)

print(df['risk_label'].value_counts())
print()
print(df['risk_label'].value_counts(normalize=True).round(3))

risk_label
at_risk         199
needs_review    118
on_track         78
Name: count, dtype: int64

risk_label
at_risk         0.504
needs_review    0.299
on_track        0.197
Name: proportion, dtype: float64


In [3]:
at_risk_df = df[df['risk_label'] == 'at_risk']
print(at_risk_df['G3'].value_counts().sort_index())

G3
0     4
1     3
2     6
3    10
4    15
5    22
6    31
7    34
8    31
9    43
Name: count, dtype: int64


In [4]:
print(df.isnull().sum().sum(), "missing values total")
print()
print(df['absences'].describe())

0 missing values total

count    395.000000
mean       4.800000
std        3.913143
min        0.000000
25%        2.000000
50%        4.000000
75%        7.000000
max       24.000000
Name: absences, dtype: float64


In [5]:
print(df.groupby('risk_label')['absences'].describe()[['count', 'mean', '50%', 'max']])

              count      mean  50%   max
risk_label                              
at_risk       199.0  5.015075  4.0  24.0
needs_review  118.0  4.601695  4.0  16.0
on_track       78.0  4.551282  4.0  20.0


In [6]:
from scipy.stats import kruskal

groups = [g['absences'].values for _, g in df.groupby('risk_label')]
stat, p = kruskal(*groups)
print("Kruskal-Wallis H:", round(stat, 3))
print("p-value:", round(p, 4))

Kruskal-Wallis H: 0.01
p-value: 0.9948


In [8]:
numeric_features = ['G1', 'G2', 'failures', 'studytime', 'goout', 'health', 'Dalc']
print(df.groupby('risk_label')[numeric_features].mean().round(2))

print()
print(pd.crosstab(df['risk_label'], df['higher'], normalize='index').round(2))

                 G1     G2  failures  studytime  goout  health  Dalc
risk_label                                                          
at_risk        6.92   6.53      0.90       2.32   3.23    2.99  1.86
needs_review  10.56  10.61      0.27       2.54   3.10    3.07  1.82
on_track      13.44  14.10      0.13       2.86   3.33    3.23  1.76

higher          no   yes
risk_label              
at_risk       0.05  0.95
needs_review  0.09  0.91
on_track      0.03  0.97


In [9]:
text_cols = df.select_dtypes(exclude='number').columns.tolist()
num_cols = df.select_dtypes(include='number').columns.tolist()

print("Text columns:", len(text_cols))
print(text_cols)
print()
print("Numeric columns:", len(num_cols))

Text columns: 18
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'risk_label']

Numeric columns: 16


In [10]:
from sklearn.model_selection import train_test_split

y = df['risk_label']

# Feature sets
features_a = [c for c in df.columns if c not in ['G2', 'G3', 'risk_label']]  # early flagging
features_b = [c for c in df.columns if c not in ['G3', 'risk_label']]         # after period 2

# Split row positions once, so both models see the same students
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=y, random_state=42
)

print("Model A features:", len(features_a))
print("Model B features:", len(features_b))
print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))
print()
print("Train class proportions:")
print(y.loc[train_idx].value_counts(normalize=True).round(3))
print()
print("Test class proportions:")
print(y.loc[test_idx].value_counts(normalize=True).round(3))

Model A features: 31
Model B features: 32
Train rows: 316 | Test rows: 79

Train class proportions:
risk_label
at_risk         0.503
needs_review    0.297
on_track        0.199
Name: proportion, dtype: float64

Test class proportions:
risk_label
at_risk         0.506
needs_review    0.304
on_track        0.190
Name: proportion, dtype: float64


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

def build_preprocessor(features):
    part = df[features]
    cat_cols = part.select_dtypes(exclude='number').columns.tolist()
    num_cols = part.select_dtypes(include='number').columns.tolist()
    return ColumnTransformer([
        ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols),
    ])

pre_a = build_preprocessor(features_a)

# Learn the rules from training rows only, then apply them to both sets
X_train_a = pre_a.fit_transform(df.loc[train_idx, features_a])
X_test_a = pre_a.transform(df.loc[test_idx, features_a])

print("Training matrix:", X_train_a.shape)
print("Test matrix:", X_test_a.shape)
print()
print(pre_a.get_feature_names_out()[:10])

Training matrix: (316, 44)
Test matrix: (79, 44)

['cat__school_MS' 'cat__sex_M' 'cat__address_U' 'cat__famsize_LE3'
 'cat__Pstatus_T' 'cat__Mjob_at_home' 'cat__Mjob_health' 'cat__Mjob_other'
 'cat__Mjob_services' 'cat__Mjob_teacher']


In [12]:
pre_b = build_preprocessor(features_b)

X_train_b = pre_b.fit_transform(df.loc[train_idx, features_b])
X_test_b = pre_b.transform(df.loc[test_idx, features_b])

print("Training matrix:", X_train_b.shape)
print("Test matrix:", X_test_b.shape)

Training matrix: (316, 45)
Test matrix: (79, 45)


In [13]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

y_train = df.loc[train_idx, 'risk_label']
y_test = df.loc[test_idx, 'risk_label']

experiments = {
    'Baseline (always most common)': (DummyClassifier(strategy='most_frequent'), X_train_a, X_test_a),
    'Logistic regression, Model A': (LogisticRegression(max_iter=1000, class_weight='balanced'), X_train_a, X_test_a),
    'Logistic regression, Model B': (LogisticRegression(max_iter=1000, class_weight='balanced'), X_train_b, X_test_b),
}

for name, (model, X_tr, X_te) in experiments.items():
    model.fit(X_tr, y_train)
    pred = model.predict(X_te)
    acc = accuracy_score(y_test, pred)
    macro_f1 = f1_score(y_test, pred, average='macro')
    print(f"{name}: accuracy={acc:.3f}, macro F1={macro_f1:.3f}")

Baseline (always most common): accuracy=0.506, macro F1=0.224
Logistic regression, Model A: accuracy=0.684, macro F1=0.657
Logistic regression, Model B: accuracy=0.709, macro F1=0.676


In [14]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1_macro']

pipe_a = make_pipeline(build_preprocessor(features_a),
                       LogisticRegression(max_iter=1000, class_weight='balanced'))
pipe_b = make_pipeline(build_preprocessor(features_b),
                       LogisticRegression(max_iter=1000, class_weight='balanced'))

res_a = cross_validate(pipe_a, df.loc[train_idx, features_a], y_train, cv=cv, scoring=scoring)
res_b = cross_validate(pipe_b, df.loc[train_idx, features_b], y_train, cv=cv, scoring=scoring)

for name, res in [('Model A', res_a), ('Model B', res_b)]:
    acc = res['test_accuracy']
    f1 = res['test_f1_macro']
    print(f"{name}: accuracy={acc.mean():.3f} (±{acc.std():.3f}), "
          f"macro F1={f1.mean():.3f} (±{f1.std():.3f})")

Model A: accuracy=0.696 (±0.021), macro F1=0.668 (±0.024)
Model B: accuracy=0.756 (±0.040), macro F1=0.739 (±0.032)


In [15]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

pred_a = cross_val_predict(pipe_a, df.loc[train_idx, features_a], y_train, cv=cv)

print(classification_report(y_train, pred_a, digits=3))
print(confusion_matrix(y_train, pred_a, labels=['at_risk', 'needs_review', 'on_track']))

              precision    recall  f1-score   support

     at_risk      0.801     0.811     0.806       159
needs_review      0.522     0.500     0.511        94
    on_track      0.677     0.698     0.688        63

    accuracy                          0.696       316
   macro avg      0.667     0.670     0.668       316
weighted avg      0.693     0.696     0.695       316

[[129  28   2]
 [ 28  47  19]
 [  4  15  44]]


In [16]:
import numpy as np

flagged_true = y_train.isin(['at_risk', 'needs_review']).values
flagged_pred = np.isin(pred_a, ['at_risk', 'needs_review'])

tp = np.sum(flagged_true & flagged_pred)
recall = tp / flagged_true.sum()
precision = tp / flagged_pred.sum()
print(f"Two-tier flag: recall={recall:.3f}, precision={precision:.3f}")
print(f"Students flagged: {flagged_pred.sum()} of {len(flagged_pred)}")

Two-tier flag: recall=0.917, precision=0.924
Students flagged: 251 of 316


In [17]:
from sklearn.metrics import classification_report, confusion_matrix

final_a = make_pipeline(build_preprocessor(features_a),
                        LogisticRegression(max_iter=1000, class_weight='balanced'))
final_a.fit(df.loc[train_idx, features_a], y_train)

test_pred = final_a.predict(df.loc[test_idx, features_a])

print(classification_report(y_test, test_pred, digits=3))
print(confusion_matrix(y_test, test_pred, labels=['at_risk', 'needs_review', 'on_track']))

flagged_true = y_test.isin(['at_risk', 'needs_review']).values
flagged_pred = np.isin(test_pred, ['at_risk', 'needs_review'])
tp = np.sum(flagged_true & flagged_pred)
print(f"Two-tier flag: recall={tp / flagged_true.sum():.3f}, "
      f"precision={tp / flagged_pred.sum():.3f}")
print(f"Students flagged: {flagged_pred.sum()} of {len(flagged_pred)}")

              precision    recall  f1-score   support

     at_risk      0.861     0.775     0.816        40
needs_review      0.480     0.500     0.490        24
    on_track      0.611     0.733     0.667        15

    accuracy                          0.684        79
   macro avg      0.651     0.669     0.657        79
weighted avg      0.698     0.684     0.688        79

[[31  9  0]
 [ 5 12  7]
 [ 0  4 11]]
Two-tier flag: recall=0.891, precision=0.934
Students flagged: 61 of 79


In [18]:
import os

joblib.dump(final_a, '../models/model_a_pipeline.joblib')
joblib.dump(features_a, '../models/model_a_features.joblib')

reloaded = joblib.load('../models/model_a_pipeline.joblib')
check = reloaded.predict(df.loc[test_idx, features_a])

print("Reloaded predictions match:", np.array_equal(check, test_pred))
print("Pipeline file size (KB):", round(os.path.getsize('../models/model_a_pipeline.joblib') / 1024, 1))
print("Feature count:", len(joblib.load('../models/model_a_features.joblib')))

Reloaded predictions match: True
Pipeline file size (KB): 8.7
Feature count: 31
